In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# results = pd.read_csv('sobel_results/detection_results.csv')
results = pd.read_csv('2024-2025_linceul_results_part2/detection_results.csv')

In [ ]:
from analyze_results import filter_rapid_changes, filter_min_max_height, apply_filters, add_moving_average, convert_px_to_metres

In [ ]:
results

In [ ]:
results['creation_date'] = pd.to_datetime(results['creation_date'], errors='coerce')

In [ ]:
filtered_results = apply_filters(results, inplace=False)
filtered_results = add_moving_average(filtered_results)

ax = results.plot(
    x='creation_date',
    y='balise_height_px',
    figsize=(12, 6),
    alpha=0.35,
    linewidth=1,
    label='Hauteur détectée',
)

filtered_results.plot(
    x='creation_date',
    y='balise_height_px_moving_average',
    ax=ax,
    linewidth=2.5,
    label=f'Hauteur filtrée (moyenne mobile {24}h + filtrage)',
)
ax.invert_yaxis()
ax.set_title('Hauteur de la balise au fil du temps')
ax.set_xlabel('Date')
ax.set_ylabel('Hauteur de la balise (px)')
ax.grid(True)
ax.legend(loc='lower right')
plt.tight_layout()

In [ ]:
dfs = []
for file in [
    '2024_linceul_results/detection_results.csv', 
    '2024-2025_linceul_results_part1/detection_results.csv',
    '2024-2025_linceul_results_part2/detection_results.csv',
]:
    print(f"Processing {file}...")
    df = pd.read_csv(file)
    df['creation_date'] = pd.to_datetime(df['creation_date'], errors='coerce')
    filtered_df = apply_filters(df, inplace=False)
    filtered_df = add_moving_average(filtered_df)
    dfs.append((file, df, filtered_df))
    
results = pd.concat([df for _, df, _ in dfs], ignore_index=True)
filtered_results = pd.concat([df for _, _, df in dfs], ignore_index=True)

ax = results.plot(
    x='creation_date',
    y='balise_height_px',
    figsize=(12, 6),
    alpha=0.35,
    linewidth=1,
    label='Hauteur détectée',
)

filtered_results.plot(
    x='creation_date',
    y='balise_height_px_moving_average',
    ax=ax,
    linewidth=2.5,
    label=f'Hauteur filtrée (moyenne mobile {24}h + filtrage)',
)

ax.invert_yaxis()
ax.set_title(f'Hauteur de la balise au fil du temps ({file})')
ax.set_xlabel('Date')
ax.set_ylabel('Hauteur de la balise (px)')
ax.grid(True)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# --- Pixel-to-metre calibration -------------------------------------------
# En prenant celle avec le moins de neige, 2024-09-03 (RCNX0486.JPG), 
# on peut voir 3,5 graduations = 35cm = 62px, soit 177px/m

px_per_metre = 177  

results_m = convert_px_to_metres(
    add_moving_average(apply_filters(results)),
    px_per_metre=px_per_metre,
)

results_m['snow_level_m'] =  results_m['balise_height_m_moving_average'].max() - results_m['balise_height_m_moving_average']

ax = results_m.plot(
    x='creation_date',
    y='snow_level_m',
    linewidth=2.5,
    label='Niveau de neige',
    figsize=(18, 6)
)
# ax.invert_yaxis()
ax.set_title('Niveau de neige estimé au fil du temps')
ax.set_xlabel('Date')
ax.set_ylabel('Niveau de neige (m)')
ax.grid(True)
plt.tight_layout()

In [ ]:
meteo = pd.read_csv('open-meteo-45.87N6.98E3946m.csv', skiprows=3)
meteo['time'] = pd.to_datetime(meteo['time'], errors='coerce')
meteo.head()

In [ ]:
plt.figure(figsize=(18, 6))

ax = results_m.plot(
    x='creation_date',
    y='snow_level_m',
    linewidth=2.5,
    label='Niveau de neige',
    figsize=(8, 5)
)

ax2 = ax.twinx()
# meteo.plot(
#     x='time',
#     y='snowfall_sum (cm)',
#     ax=ax2,
#     color='orange',
#     label='Précipitations de neige (cm)'
# )
# ax2.legend('lower left')

meteo.plot(
    x='time',
    y='precipitation_sum (mm)',
    ax=ax2,
    color='red',
    label='Précipitations totales (mm)'
)
ax2.legend(loc='upper right')

ax.set_title('Niveau de neige estimé au fil du temps')
ax.set_xlabel('Date')
ax.set_ylabel('Niveau de neige (m)')
ax2.set_ylabel('Précipitations (mm)')
ax.grid(True)
plt.tight_layout()
plt.savefig('snow_level_and_precipitations.png', dpi=300)

In [ ]:
meteo[50:]